# Measuring league strength from transfers

*Question, Intuition, Math, Code, Assumptions, How it breaks*

## 1. Question

A player's season is scored against the other players in **his** league. So a
2.0 in Ligue 1 and a 2.0 in the Premier League both mean "two standard
deviations above your peers", except the peers are not equally good.

If I pool a career across leagues, which I have to do the moment anyone
transfers, how do I avoid rewarding whoever happened to play in the weakest
division?

## 2. Intuition

**A player who changes league is the same footballer on both sides of the
move.** So whatever happens to his score across that move is a measurement of
the difference between the two leagues.

One transfer is a noisy measurement. Thousands of transfers, forming a connected
graph over 25 years, pin down the whole system. It is the same device that makes
chess ratings comparable across separate rating pools.

## 3. Math

For a move from league $a$ to league $b$:

$$\Delta z = \alpha + \beta \cdot \text{age} + \lambda_a - \lambda_b + \varepsilon$$

- $\lambda$ are the league strengths I want. They are identified only **up to a
  constant**, so one league gets pinned at zero.
- $\alpha$ is a global adaptation term. Settling into a new league costs
  something on average, and that cost is not league strength.
- $\beta$ controls for age, since players move at different career stages and
  ordinary decline would otherwise get misread as league difficulty.

Fitted by weighted least squares, weighting each move by the smaller of the two
seasons' minutes.

In [ ]:
import warnings

import matplotlib
import pandas as pd

from gambeta import bridge

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
offsets = pd.read_parquet(f"{SAMPLE}/league_offsets.parquet")
keepers = pd.read_parquet(f"{SAMPLE}/keeper_ranking.parquet")

# One offset is estimated per five-season block and then written onto all five
# of that block's season rows, so summing the move count down the season column
# would multiply it by five. Deduplicate to the block first.
per_block = offsets.assign(block=offsets["season"].map(bridge.era_block)).drop_duplicates(
    ["league", "block"]
)
summary = (
    per_block.groupby("league")
    .agg(offset=("offset", "mean"), moves=("moves", "sum"))
    .sort_values("offset", ascending=False)
)
summary.round(3)

## Is England best, or just first?

The Premier League sits at 0.000 and everything else is negative, which looks
like a verdict. It is not. **The reference league is pinned by construction**,
because the offsets are identified only up to a constant. Add 0.5 to every
league and not one transfer in the data would look any different.

So the fair challenge is this: does England lead because it is stronger, or
because I happened to pin it there? There are two ways to answer, and they
agree.

**First, move the pin.** If England's position were an artefact of being the
reference, then choosing a different reference would change the gaps.

In [ ]:
import dataclasses

from gambeta import kit, level, needs

cfg = kit.load()

# The published `season_score` is measured AFTER the offsets are applied, so
# using it here would be circular -- the effect we are trying to see has already
# been subtracted out. Rebuild the pre-bridge score: z-score each requirement
# within its own (league, season), then average.
keys = needs.season_keys(needs.OUTFIELD)
pre = level.zscore(seasons, keys)
pre["score"] = pre[[f"{k}_z" for k in keys]].mean(axis=1)

moves = bridge.find_moves(pre)
present = sorted(seasons["league"].dropna().unique())

pins = {}
for pin in present:
    reordered = dataclasses.replace(cfg, leagues=(pin, *[lg for lg in present if lg != pin]))
    solved = bridge.solve_offsets(moves, reordered, leagues=present).groupby("league")["offset"]
    pins[f"pin {pin.split('-')[0]}"] = solved.mean() - solved.mean()["ENG-Premier League"]

comparison = pd.DataFrame(pins)
spread = (comparison.max(axis=1) - comparison.min(axis=1)).max()
print(f"{len(moves):,} transfers. Gaps relative to England, under every choice of pin:\n")
print(comparison.round(3).to_string())
print(f"\nlargest disagreement between pinnings: {spread:.3f}")

The gaps agree to within 0.013 whichever league gets pinned. That is a
twentieth of the England-to-France gap, and it comes from the least-squares fit
having slightly different collinearity to resolve each time. The pin sets where
zero sits. It does not decide the ordering.

**Second, skip the model entirely.** The regression exists to pool noisy
evidence, but you can read the raw evidence straight off the transfers. What
happens to a player's score when he moves into England, against what happens
when he moves out?

In [ ]:
ENG = "ENG-Premier League"
rows = []
for other in [lg for lg in present if lg != ENG]:
    into = moves[(moves["from_league"] == other) & (moves["to_league"] == ENG)]["delta"]
    away = moves[(moves["from_league"] == ENG) & (moves["to_league"] == other)]["delta"]
    rows.append(
        {
            "league": other.split("-")[0],
            "moves in": len(into),
            "score change joining England": round(into.mean(), 3),
            "moves out": len(away),
            "score change leaving England": round(away.mean(), 3),
        }
    )

pd.DataFrame(rows).set_index("league")

Every row runs the same way. Joining England costs a player score, leaving
England gains it, from the same four leagues in both directions. **That
asymmetry is the entire basis of the claim.** It is visible without any
regression at all, and the regression's only job is to pool it across every pair
at once while controlling for age and for the common cost of moving anywhere.

One honest caveat before I get to the corroboration. This measures *where
scoring output is harder to produce*, not league quality in the abstract. Those
two things are related and they are not identical. A league with tighter
defences and lower scoring would look "stronger" here even if its best players
were no better. Nothing in this data separates those two explanations.

In [ ]:
offsets["era"] = offsets["season"].str[:2].astype(int) // 5
era = offsets.pivot_table(index="league", columns="era", values="offset", aggfunc="mean")
era.columns = ["2000-04", "2005-09", "2010-14", "2015-19", "2020-24"]
era.round(2)

## 4. Code, and the corroboration that matters

Look at the era table above. The gaps are small in 2000-04 and they widen
sharply from 2005 onwards.

That is the Premier League's financial ascent. And **nothing about money, TV
deals or transfer fees is anywhere in this model.** It was recovered purely from
players changing league and their scores changing with them. When an estimate
reproduces a known historical pattern it never had access to, that is real
evidence the method works.

In [ ]:
endpoints = per_block.groupby("league")["moves"].sum().sort_values(ascending=False)
print("move endpoints involving each league:")
for lg, n in endpoints.items():
    print(f"  {lg:<22} {n:>5,}")

print(f"\ntotal endpoints {endpoints.sum():,}, so {endpoints.sum() // 2:,} distinct transfers")
print("(every move has two ends, and is counted once at each).")
print("\nAn offset backed by 200 moves deserves less trust than one backed by 900,")
print("which is why the count is published alongside the estimate.")

## 5. Assumptions

1. **A transferring player is unchanged by the move**, apart from age and a
   common adaptation effect. Injuries, motivation and tactical fit all violate
   this individually. I am hoping they average out.
2. **Transfers are not selective in a way that correlates with the gap.** They
   almost certainly are. Players usually move *up* when they excel and *down*
   when they decline, which is exactly the kind of selection that biases this.
3. **League strength is constant within a five-season block.** A compromise:
   per-season offsets would be badly identified from the handful of moves some
   pairs see in a single year.

## 6. How it breaks

There is a failure mode here that stays invisible unless you go looking for it.
**With transfers in only one direction between two leagues, the adaptation term
and the league offset are perfectly collinear.**

Both apply to every move. The only thing separating them is how they behave
under a sign flip: the offset flips when the direction flips, adaptation does
not. If everybody moves one way, no amount of data will separate them, and the
solver will just split the difference arbitrarily.

In [ ]:
from gambeta import bridge, kit

cfg = kit.load()


def moves_between(gap, n, both_ways):
    rows = []
    for i in range(n):
        rows += [
            {
                "player_id": f"o{i}",
                "league": "FRA-Ligue 1",
                "season": "0102",
                "score": 1.0 + gap,
                "minutes": 3000,
                "age": 25.0,
            },
            {
                "player_id": f"o{i}",
                "league": "ENG-Premier League",
                "season": "0203",
                "score": 1.0,
                "minutes": 3000,
                "age": 26.0,
            },
        ]
        if both_ways:
            rows += [
                {
                    "player_id": f"i{i}",
                    "league": "ENG-Premier League",
                    "season": "0102",
                    "score": 0.5,
                    "minutes": 3000,
                    "age": 25.0,
                },
                {
                    "player_id": f"i{i}",
                    "league": "FRA-Ligue 1",
                    "season": "0203",
                    "score": 0.5 + gap,
                    "minutes": 3000,
                    "age": 26.0,
                },
            ]
    return pd.DataFrame(rows)


pair = ["ENG-Premier League", "FRA-Ligue 1"]
for both in (True, False):
    m = bridge.find_moves(moves_between(0.5, 40, both))
    o = bridge.solve_offsets(m, cfg, leagues=pair)
    est = o[(o["league"] == "FRA-Ligue 1") & (o["season"] == "0102")]["offset"].item()
    label = "both directions" if both else "one direction only"
    print(f"  {label:<20} true gap -0.50, estimated {est:+.2f}")

With traffic both ways the true gap gets recovered. With one-way traffic, half
of it is silently absorbed into the adaptation term and the league looks
stronger than it is.

Real transfer data flows both ways between all five leagues, which is what makes
these estimates identifiable at all. But a league with mostly outbound moves, a
selling league, would be systematically mis-measured, and that is a live risk
rather than a hypothetical one. Ligue 1 is the closest thing here to that case.
It sells more than it buys, and it lands lowest of the five.